# Coleta — vincular Proposições aos Parlamentares (autores)

Este notebook liga cada **proposição (PL)** ao(s) **deputado(s)** que a assinaram, e salva em
`proposicoes_parlamentares.csv`. Esse vínculo é pré-requisito da Fase 6 (cruzar discursos ×
proposições por deputado).

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Como funciona:** em vez de perguntar o autor de cada uma das ~19 mil proposições (lento),
  invertemos a busca — para cada **deputado atual** pedimos as PLs que ele assinou, via o
  parâmetro `idDeputadoAutor`. Mais rápido (~10 min) e sem depender de casar nomes.
- **Decisão:** só consultamos os **513 deputados atuais**; autores de mandato encerrado e
  comissões/órgãos não entram (não teremos discursos deles). A célula final reporta a
  **cobertura** (% das PLs assinadas por deputado atual).
- **Importante:** os dados são coletados ao vivo da fonte oficial. **Não usamos base pronta.**

> Pré-requisitos na pasta: `parlamentares.csv` (gerado por `02_coleta_parlamentares.ipynb`) e
> `proposicoes_temas.csv` (gerado por `01_coleta_dados.ipynb`).

In [10]:
import requests   # para acessar a API pela internet
import time       # para dar pequenas pausas entre as chamadas
import pandas as pd  # para organizar os dados em tabela

BASE = "https://dadosabertos.camara.leg.br/api/v2"

def chamar_api(caminho, parametros=None):
    """Faz uma chamada na API e devolve o resultado (em formato JSON).
    Tenta ate 3 vezes caso o servidor falhe."""
    for tentativa in range(3):
        try:
            resposta = requests.get(
                BASE + caminho,
                params=parametros,
                headers={"Accept": "application/json"},
                timeout=30,
            )
            if resposta.status_code == 200:
                return resposta.json()
        except requests.RequestException:
            pass
        time.sleep(1)   # espera 1 segundo e tenta de novo
    return None

print("Pronto! A funcao chamar_api esta carregada.")

Pronto! A funcao chamar_api esta carregada.


In [11]:
# Em vez de perguntar "quem e o autor?" para cada uma das ~19 mil proposicoes
# (lento: ~2,7h), invertemos a busca: para cada deputado ATUAL, pedimos as PLs
# que ele assinou, usando o parametro idDeputadoAutor. Sao ~2 mil chamadas (~10 min).
#
# Consequencia (decisao do projeto): so consultamos os 513 deputados atuais, entao
# autores de mandato encerrado e comissoes/orgaos nao entram -- e o que queremos,
# pois nao teremos discursos deles para o cruzamento.

deps = pd.read_csv("parlamentares.csv", sep=";", dtype=str)
deps["nome"] = deps["nome"].str.strip()

# so guardamos pares cuja proposicao esta na nossa base com tema (alinhamento garantido)
props_validas = set(pd.read_csv("proposicoes_temas.csv", dtype=str)["id"])

ANOS = [2023, 2024, 2025, 2026]

linhas = []   # cada item: uma PL assinada por um deputado atual
for n, (_, dep) in enumerate(deps.iterrows(), start=1):
    for ano in ANOS:
        pagina = 1
        while True:
            r = chamar_api("/proposicoes", {
                "idDeputadoAutor": dep["id"],
                "siglaTipo": "PL",
                "ano": ano,
                "itens": 100,
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "id",
            })
            # sem resposta ou pagina vazia => acabou este deputado/ano
            if not r or not r["dados"]:
                break
            for p in r["dados"]:
                pid = str(p["id"])
                if pid in props_validas:
                    linhas.append({
                        "proposicao_id": pid,
                        "id_autor": dep["id"],
                        "nome_autor": dep["nome"],
                    })
            pagina += 1
            time.sleep(0.2)   # pausa para nao sobrecarregar o servidor
    if n % 50 == 0:
        print(f"  ... {n}/{len(deps)} deputados processados ({len(linhas)} pares ate agora)")

df_autores_proposicoes = pd.DataFrame(linhas).drop_duplicates()
df_autores_proposicoes.to_csv("proposicoes_parlamentares.csv",
                              index=False, sep=";", encoding="utf-8")
print(f"\nTotal de pares (proposicao, deputado atual): {len(df_autores_proposicoes)}")
print("Arquivo salvo: proposicoes_parlamentares.csv")

  ... 50/513 deputados processados (2331 pares ate agora)
  ... 100/513 deputados processados (4853 pares ate agora)
  ... 150/513 deputados processados (6996 pares ate agora)
  ... 200/513 deputados processados (10006 pares ate agora)
  ... 250/513 deputados processados (11406 pares ate agora)
  ... 300/513 deputados processados (13921 pares ate agora)
  ... 350/513 deputados processados (16344 pares ate agora)
  ... 400/513 deputados processados (17911 pares ate agora)
  ... 450/513 deputados processados (19965 pares ate agora)
  ... 500/513 deputados processados (21888 pares ate agora)

Total de pares (proposicao, deputado atual): 22345
Arquivo salvo: proposicoes_parlamentares.csv


In [12]:
# Cobertura e checagem de plausibilidade
total_props = len(props_validas)
props_cobertas = df_autores_proposicoes["proposicao_id"].nunique()
print(f"Proposicoes com tema na base....: {total_props}")
print(f"Proposicoes com >=1 autor atual.: {props_cobertas} "
      f"({100*props_cobertas/total_props:.1f}% de cobertura)")
print(f"Deputados atuais com >=1 PL.....: {df_autores_proposicoes['id_autor'].nunique()} de 513")

print("\nTop 10 deputados por nº de PLs assinadas (checagem de plausibilidade):")
top = (df_autores_proposicoes.groupby("nome_autor")["proposicao_id"]
       .nunique().sort_values(ascending=False).head(10))
print(top.to_string())

Proposicoes com tema na base....: 19354
Proposicoes com >=1 autor atual.: 17079 (88.2% de cobertura)
Deputados atuais com >=1 PL.....: 495 de 513

Top 10 deputados por nº de PLs assinadas (checagem de plausibilidade):
nome_autor
Amom Mandel             745
Duda Ramos              723
Marcos Tavares          489
Jonas Donizette         442
Duarte Jr.              252
José Medeiros           246
Laura Carneiro          233
Kim Kataguiri           233
Cabo Gilberto Silva     203
Evair Vieira de Melo    192
